# MRH Analysis — position embeddings & empirical evidence for the Minkowski Representation Hypothesis

Covers `PAPER_NOTES.md` Section 12 (position-embedding analysis) and Section 14 (MRH
empirical evidence: k-NN geodesics, Archetypal Analysis vs. SAE, block structure).
Requires `rabbit_hull.ipynb` (Phase 1) to have been run first.

## 0. Setup

In [ ]:
import os, sys

# Anchors on markers unique to this repo so `import config` always finds THIS
# project's src/, regardless of how/where Colab's CWD ended up (upload, git
# clone, Drive mount, ...) — sys.path.append("src") alone only works if CWD
# already happens to be the repo root, which Colab doesn't guarantee.
_MARKERS = (os.path.join("src", "config.py"), os.path.join("src", "sae.py"))
_here = os.getcwd()
_candidates = [
    _here,
    os.path.join(_here, "rabbit_hull"),
    "/content/rabbit_hull",
    os.path.dirname(_here),
]
for _root in _candidates:
    if all(os.path.isfile(os.path.join(_root, m)) for m in _MARKERS):
        os.chdir(_root)
        sys.path.insert(0, os.path.join(_root, "src"))
        break
else:
    raise RuntimeError(
        "Could not locate the repo root (needs src/config.py and src/sae.py). "
        "cd into the rabbit_hull folder first (e.g. %cd /content/rabbit_hull), "
        "then re-run this cell."
    )

print("Repo root:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from config import CONFIG
import utils
import data_utils
import model_utils
import sae as sae_module
import position_utils
import mrh_utils

print(CONFIG)

## 1. Load cached SAE checkpoint and analysis-set activations

In [ ]:
centroids_path = Path(CONFIG["cache_dir"]) / "centroids.pt"
sae_path = Path(CONFIG["cache_dir"]) / "checkpoints" / "sae.pt"
if not (centroids_path.exists() and sae_path.exists()):
    raise RuntimeError(
        "No cached centroids/SAE checkpoint found — run rabbit_hull.ipynb (Phase 1) first."
    )
centroids = torch.load(centroids_path, weights_only=True)
sae = sae_module.load_checkpoint(centroids, CONFIG)
sae.to(CONFIG["device"])
sae.eval()

model, processor = model_utils.load_model(CONFIG)

analysis_acts_path = Path(CONFIG["cache_dir"]) / "activations" / "analysis.pt"
if analysis_acts_path.exists():
    print(f"Loading cached {analysis_acts_path.name}…")
    analysis_acts = torch.load(analysis_acts_path, weights_only=True)
else:
    val_dataset = data_utils.load_imagenet_val(CONFIG)
    analysis_images, _ = data_utils.load_analysis_images(val_dataset, CONFIG)
    analysis_acts = utils.cached(
        analysis_acts_path,
        lambda: model_utils.extract_activations(analysis_images, model, processor, CONFIG),
    )

rng = np.random.RandomState(CONFIG["random_state"])

## 2. Position-embedding analysis

Per-layer positional basis (direct-averaging + linear-classifier methods, §12) across all
of DINOv2's hidden-state layers. Uses `model_utils.extract_all_layer_activations` on a
fresh `CONFIG["n_position"]`-image sample — deliberately **not** cached to Drive (~10GB
at these defaults; cheaper to recompute per session than to store, see that function's
docstring). Lower `CONFIG["n_position"]` if this doesn't fit your Colab runtime's RAM.

In [ ]:
val_dataset = data_utils.load_imagenet_val(CONFIG)
position_idx = rng.choice(len(val_dataset), CONFIG["n_position"], replace=False)
position_images = [val_dataset[int(i)]["image"].convert("RGB") for i in position_idx]

all_layer_acts = model_utils.extract_all_layer_activations(
    position_images, model, processor, CONFIG
).float().numpy()
n_layers = all_layer_acts.shape[1]

In [ ]:
accuracies, stable_ranks, position_bases = [], [], []
for layer in range(n_layers):
    layer_acts = all_layer_acts[:, layer, :, :]
    P, acc = position_utils.train_position_classifier(layer_acts, CONFIG)
    position_bases.append(P)
    accuracies.append(acc)
    stable_ranks.append(position_utils.stable_rank(P))
    print(f"Layer {layer}: position-decoding accuracy={acc:.4f}  stable_rank(P)={stable_ranks[-1]:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(accuracies); axes[0].set_xlabel("layer"); axes[0].set_ylabel("position-decoding accuracy")
axes[1].plot(stable_ranks); axes[1].set_xlabel("layer"); axes[1].set_ylabel("stable rank of P")
plt.suptitle("Position decodability and positional-subspace rank across layers (Fig 14)")
plt.show()

## 3. Position removal check

Project the last layer's tokens orthogonal to its positional basis, then compare
per-image PCA structure before/after (Appendix I.2, Fig 27) — checking that the
interpolative/local-connectedness geometry isn't merely inherited from positional
encoding.

In [ ]:
layer_for_removal = n_layers - 1
P_last = position_bases[layer_for_removal]

example_idx = 0
tokens_before = all_layer_acts[example_idx, layer_for_removal, utils.SPATIAL_SLICE, :]
tokens_after = position_utils.remove_positional_subspace(tokens_before, P_last)

n_before = position_utils.per_image_pca_components(tokens_before)
n_after = position_utils.per_image_pca_components(tokens_after)
print(f"PCA components for 95% variance — before removal: {n_before}, after removal: {n_after}")

for name, tokens in [("before", tokens_before), ("after", tokens_after)]:
    rgb = PCA(n_components=3).fit_transform(tokens)
    rgb = (rgb - rgb.min(axis=0)) / (rgb.max(axis=0) - rgb.min(axis=0) + 1e-8)
    plt.figure(figsize=(3, 3))
    plt.imshow(rgb.reshape(16, 16, 3))
    plt.title(f"Top-3 PCA RGB ({name} position removal)")
    plt.axis("off")
    plt.show()

## 4. k-NN geodesics vs. linear interpolation

Cosine-distance k-NN graph over a sample of last-layer analysis tokens; compares
straight-line interpolation between two tokens against the piecewise-linear graph
geodesic, via distance-to-nearest-data-sample along each path (Fig 17 left).

In [ ]:
flat_tokens = analysis_acts.float().numpy().reshape(-1, CONFIG["d_model"])
sample_idx = rng.choice(len(flat_tokens), CONFIG["n_mrh_tokens"], replace=False)
mrh_tokens = flat_tokens[sample_idx]

graph = mrh_utils.build_knn_graph(mrh_tokens, CONFIG["knn_k"])

src = 0
path_nodes = None
for _ in range(20):
    dst = int(rng.integers(1, len(mrh_tokens)))
    try:
        path_nodes = mrh_utils.graph_geodesic_path(graph, src, dst)
        break
    except ValueError:
        continue
if path_nodes is None:
    raise RuntimeError("Could not find a connected src/dst pair — try increasing CONFIG['knn_k'].")

In [ ]:
steps = 11
linear_points = mrh_utils.path_points(mrh_tokens[[src, dst]], steps)
geodesic_points = mrh_utils.path_points(mrh_tokens[path_nodes], steps)

linear_dist = mrh_utils.distance_to_manifold(linear_points, mrh_tokens)
geodesic_dist = mrh_utils.distance_to_manifold(geodesic_points, mrh_tokens)

plt.plot(np.linspace(0, 1, steps), linear_dist, label="linear interpolation")
plt.plot(np.linspace(0, 1, steps), geodesic_dist, label="graph geodesic")
plt.xlabel("interpolation step"); plt.ylabel("distance to nearest data sample")
plt.legend(); plt.title("On-manifold geodesics vs. linear interpolation (Fig 17 left)")
plt.show()

## 5. Archetypal Analysis vs. SAE

Run per-image (not dataset-wide — this is what makes "~10 archetypes per image"
tractable): for `CONFIG["n_aa_images"]` images, sweep `CONFIG["aa_archetype_counts"]` and
compare AA's reconstruction MSE against the trained SAE's own per-image reconstruction
MSE (Fig 17 middle).

In [ ]:
aa_image_idx = rng.choice(analysis_acts.shape[0], CONFIG["n_aa_images"], replace=False)

aa_curves, sae_mses = [], []
for image_idx in aa_image_idx:
    tokens = analysis_acts[image_idx].float().numpy()  # (261, d_model)
    tokens_torch = torch.tensor(tokens, dtype=torch.float32, device=CONFIG["device"])
    with torch.no_grad():
        z_dense = sae_module.encode_dense(sae, tokens_torch)
        recon = (z_dense @ sae.dictionary()).cpu().numpy()
    sae_mses.append(float(((recon - tokens) ** 2).mean()))

    curve = [
        mrh_utils.archetypal_analysis(tokens, n_arch, CONFIG["aa_iters"], rng)[3]
        for n_arch in CONFIG["aa_archetype_counts"]
    ]
    aa_curves.append(curve)

mean_aa_curve = np.array(aa_curves).mean(axis=0)
mean_sae_mse = np.mean(sae_mses)

plt.plot(CONFIG["aa_archetype_counts"], mean_aa_curve, marker="o", label="Archetypal Analysis")
plt.axhline(mean_sae_mse, color="k", linestyle="--", label="SAE reconstruction MSE")
plt.xlabel("# archetypes"); plt.ylabel("reconstruction MSE"); plt.legend()
plt.title("AA vs. SAE reconstruction (Fig 17 middle)")
plt.show()

## 6. Block structure

Pool archetype codings (at the largest swept archetype count) across the same AA images,
hierarchically cluster/reorder, and visualize the reordered Gram matrix — emergent
block-diagonal structure without any explicit tile-boundary specification (Fig 17
right).

In [ ]:
block_archetype_count = CONFIG["aa_archetype_counts"][-1]
codings = []
for image_idx in aa_image_idx:
    tokens = analysis_acts[image_idx].float().numpy()
    _, _, A, _ = mrh_utils.archetypal_analysis(tokens, block_archetype_count, CONFIG["aa_iters"], rng)
    codings.append(A)
pooled_coding = np.concatenate(codings, axis=0)  # (n_aa_images * 261, block_archetype_count)

order = mrh_utils.block_structure(pooled_coding)
gram = pooled_coding @ pooled_coding.T
reordered = gram[np.ix_(order, order)]

plt.imshow(reordered, cmap="viridis")
plt.title("Archetype-coding Gram matrix, hierarchically reordered (Fig 17 right)")
plt.colorbar()
plt.show()

## Summary

Across `rabbit_hull.ipynb` + `classification_segmentation.ipynb` +
`dictionary_geometry.ipynb` + this notebook, `PAPER_NOTES.md` §2, §4, §6-12, §14 are now
implemented at reduced scale (`n_atoms=1000` vs. 32,000, and further reductions
per-notebook — see each notebook's closing cell and `README.md`'s differences table).

**Intentionally out of scope**: depth estimation (§7's third downstream task — see
`README.md`), and the DinoVision visualization tool (§16, a UI artifact rather than an
analysis to reimplement).